# 02. От анализа к действию: подключаем Jenkins

## Что агент пока не умеет

После `01` агент умеет исследовать локальный workspace, но не может взаимодействовать с внешними инженерными системами.

> Tool — это контролируемый контракт между рассуждением модели и обычным детерминированным кодом.


## Разделение ответственности

```text
Модель:
- понимает намерение пользователя;
- выбирает действие;
- формирует параметры;
- интерпретирует результат.

Python connector:
- выполняет HTTP-запрос;
- знает authentication;
- проверяет входные данные;
- нормализует ответ;
- обрабатывает ошибки.
```

Модель не видит весь исходный код connector. Она видит описание способности и JSON-схему входа.


## Реальные Jenkins capabilities в этом стенде

```text
read:
- job metadata;
- last build reference/result.

write:
- trigger build.
```

Dry-run остаётся диагностикой, но кульминация главы — настоящий build request.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import REPO_ROOT, print_stage_context, register_graphs, write_text
from connectors.jenkins import get_jenkins_job_info, trigger_jenkins_job

print_stage_context()
print(get_jenkins_job_info.name)
print(get_jenkins_job_info.description)
print(trigger_jenkins_job.args_schema.model_json_schema())


In [ ]:
# Diagnostic preview: useful before the live POST, not the main demo.
print(trigger_jenkins_job.invoke({
    "parameters": {"OPENCLAW_SMOKE": "true"},
    "dry_run": True,
}))


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n\n\ndef _backend(*, require_shell: bool = False):\n    root = _workspace_root()\n    shell_enabled = os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\"\n    if require_shell and not shell_enabled:\n        raise RuntimeError(\"OPENCLAW_ENABLE_LOCAL_SHELL=1 is required for this stage\")\n    if shell_enabled:\n        from deepagents.backends import LocalShellBackend\n\n        return LocalShellBackend(\n            root_dir=root,\n            virtual_mode=True,\n            inherit_env=False,\n            timeout=120,\n            max_output_bytes=80_000,\n        )\n\n    from deepagents.backends import FilesystemBackend\n\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 02 with Jenkins tools. Respond in the user's language; default to Russian.\nSeparate read actions from write actions. If the user explicitly asks for a real Jenkins build, call trigger_jenkins_job with dry_run=False.\nReturn normalized operational summaries, not raw dumps.\n\"\"\"\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=JENKINS_TOOLS,\n    system_prompt=BASE_PROMPT,\n    backend=_backend(),\n)\n"
print(write_text("agents/openclaw_02_jenkins_tools.py", ENTRYPOINT).relative_to(REPO_ROOT))
print(register_graphs({
    "openclaw_02": "./agents/openclaw_02_jenkins_tools.py:agent",
}).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Сначала проверь Jenkins job и назови статус последней сборки. Затем запусти smoke build с параметром OPENCLAW_SMOKE=true. Это реальный запуск.
```

### Ожидаемое поведение

1. Агент вызывает `get_jenkins_job_info`.
2. Затем вызывает `trigger_jenkins_job` с `dry_run=false`.
3. Ответ содержит `ok`, `status_code` и `queue_url`, если Jenkins принял запрос.

### Что изменилось относительно предыдущего этапа

Появилось внешнее инженерное действие через контролируемый Python tool contract.

### Текущее ограничение

Человек всё ещё находится в LangGraph Studio; агент пока не имеет привычного messenger-first интерфейса.
